# 주택 가격 예측하기

메인퀘스트에 오신 것을 환영합니다!

이번 퀘스트의 목표는 여러 특성들을 종합적으로 고려하여 가장 주택 가격을 잘 예측하는 회귀모델을 만드는 것입니다.

- 예측할 값(y): SalePrice
- 평가지표: RMSE

예시 형식:
~~~
Id,SalePrice
1461,169000.1
1462,187724.1233
1463,175221
etc.
~~~

In [ ]:
# %pip install scikit-learn==1.9.1

In [1]:
import sklearn
print(sklearn.__version__)   # 1.9.0이 나와야 함

1.9.1


### 데이터 확인 및 불러오기

In [2]:
import numpy as np
import pandas as pd
import os

# 교차 검증과 모델에 필요한 도구
from sklearn.model_selection import KFold, cross_val_score
from sklearn.linear_model import LinearRegression


for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

In [3]:
train_df = pd.read_csv('./train.csv')
test_df = pd.read_csv('./test.csv')

In [4]:
# ===== 5겹 교차 검증 함수 =====
# 이유: train을 한 번만 나누면 검증용에 어떤 집이 뽑히느냐에 따라 점수가 크게 흔들린다.
#      5조각으로 나눠 조각마다 한 번씩 검증용을 맡기고 5개 점수를 평균 내면
#      운의 영향이 줄어, 방법이 정말 좋아졌는지 비교할 수 있다.

kf = KFold(
    n_splits=5,        # 5조각으로 나눈다
    shuffle=True,      # 데이터가 정렬돼 있을 수 있으므로 섞은 뒤 나눈다
    random_state=42    # 실험마다 같은 5조각으로 나눠야 점수 차이가 방법 차이에서만 나온다
)

def cv_rmse(model, X, y):
    # scikit-learn은 "클수록 좋다" 규칙이라 RMSE에 마이너스를 붙여 돌려준다 → 앞의 -로 다시 양수로
    scores = -cross_val_score(model, X, y, cv=kf, scoring="neg_root_mean_squared_error")
    return scores.mean()

In [5]:
train_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1168 entries, 0 to 1167
Data columns (total 81 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Id             1168 non-null   int64  
 1   MSSubClass     1168 non-null   int64  
 2   MSZoning       1168 non-null   str    
 3   LotFrontage    950 non-null    float64
 4   LotArea        1168 non-null   int64  
 5   Street         1168 non-null   str    
 6   Alley          70 non-null     str    
 7   LotShape       1168 non-null   str    
 8   LandContour    1168 non-null   str    
 9   Utilities      1168 non-null   str    
 10  LotConfig      1168 non-null   str    
 11  LandSlope      1168 non-null   str    
 12  Neighborhood   1168 non-null   str    
 13  Condition1     1168 non-null   str    
 14  Condition2     1168 non-null   str    
 15  BldgType       1168 non-null   str    
 16  HouseStyle     1168 non-null   str    
 17  OverallQual    1168 non-null   int64  
 18  OverallCond    1168

특성이 너무 많습니다! 어떻게 하면 좋을까요?

참고로 각각의 특성은 다음을 나타냅니다.

| 변수명               | 설명 (한국어)                         |
| ----------------- | -------------------------------- |
| **SalePrice**     | 주택의 판매 가격(달러). 예측해야 하는 타겟 변수     |
| **MSSubClass**    | 건물 등급(건물 종류 분류)                  |
| **MSZoning**      | 토지 용도 구분(일반적인 구역 분류)             |
| **LotFrontage**   | 도로에 접한 토지 길이 (피트 단위)             |
| **LotArea**       | 토지 면적 (평방 피트)                    |
| **Street**        | 도로 접근 방식                         |
| **Alley**         | 골목(후면 도로) 접근 방식                  |
| **LotShape**      | 토지의 전반적인 모양                      |
| **LandContour**   | 토지의 평탄도                          |
| **Utilities**     | 사용 가능한 공공시설(전기, 수도 등)            |
| **LotConfig**     | 토지 배치 형태                         |
| **LandSlope**     | 토지의 경사도                          |
| **Neighborhood**  | Ames 시 내의 물리적 위치(주변 지역)          |
| **Condition1**    | 주요 도로·철도에 대한 근접도                 |
| **Condition2**    | 두 번째 주요 도로·철도에 대한 근접도            |
| **BldgType**      | 주거 건물 유형 (단독, 다세대 등)             |
| **HouseStyle**    | 주택의 건축 양식                        |
| **OverallQual**   | 자재 및 마감 품질 종합 평가                 |
| **OverallCond**   | 주택의 전반적인 상태 평가                   |
| **YearBuilt**     | 최초 건축 연도                         |
| **YearRemodAdd**  | 리모델링/개조 연도                       |
| **RoofStyle**     | 지붕 스타일                           |
| **RoofMatl**      | 지붕 재료                            |
| **Exterior1st**   | 외장 마감재(주 재료)                     |
| **Exterior2nd**   | 외장 마감재(보조 재료, 2번째)               |
| **MasVnrType**    | 석조 베니어 타입                        |
| **MasVnrArea**    | 석조 베니어 면적 (평방 피트)                |
| **ExterQual**     | 외장재 품질 평가                        |
| **ExterCond**     | 외장재 상태 평가                        |
| **Foundation**    | 기초(Foundation) 유형                |
| **BsmtQual**      | 지하실 높이 평가                        |
| **BsmtCond**      | 지하실 상태 평가                        |
| **BsmtExposure**  | 지하실 외부 노출 여부(출입구/창문 등)           |
| **BsmtFinType1**  | 주요 지하실 마감 구역 유형                  |
| **BsmtFinSF1**    | 주요 지하실 마감 면적 (평방 피트)             |
| **BsmtFinType2**  | 보조 지하실 마감 구역 유형                  |
| **BsmtFinSF2**    | 보조 지하실 마감 면적 (평방 피트)             |
| **BsmtUnfSF**     | 미완성 지하실 면적 (평방 피트)               |
| **TotalBsmtSF**   | 지하실 전체 면적 (평방 피트)                |
| **Heating**       | 난방 시스템 종류                        |
| **HeatingQC**     | 난방 품질 및 상태                       |
| **CentralAir**    | 중앙 에어컨 설치 여부                     |
| **Electrical**    | 전기 시스템 종류                        |
| **1stFlrSF**      | 1층 면적 (평방 피트)                    |
| **2ndFlrSF**      | 2층 면적 (평방 피트)                    |
| **LowQualFinSF**  | 저품질 마감 공간 면적 (전체 층 합산)           |
| **GrLivArea**     | 지상(Above grade) 생활 공간 면적 (평방 피트) |
| **BsmtFullBath**  | 지하실 완비 욕실 수                      |
| **BsmtHalfBath**  | 지하실 반 욕실 수                       |
| **FullBath**      | 지상 완비 욕실 수                       |
| **HalfBath**      | 지상 반 욕실 수                        |
| **Bedroom**       | 지하실 제외 침실 수                      |
| **Kitchen**       | 주방 수                             |
| **KitchenQual**   | 주방 품질                            |
| **TotRmsAbvGrd**  | 지상 전체 방 개수 (욕실 제외)               |
| **Functional**    | 주택 기능성 평가                        |
| **Fireplaces**    | 벽난로 개수                           |
| **FireplaceQu**   | 벽난로 품질                           |
| **GarageType**    | 차고 위치(유형)                        |
| **GarageYrBlt**   | 차고 건축 연도                         |
| **GarageFinish**  | 차고 내부 마감 상태                      |
| **GarageCars**    | 차고 크기 (주차 가능 차량 수)               |
| **GarageArea**    | 차고 면적 (평방 피트)                    |
| **GarageQual**    | 차고 품질                            |
| **GarageCond**    | 차고 상태                            |
| **PavedDrive**    | 포장 진입로 여부                        |
| **WoodDeckSF**    | 목재 데크 면적 (평방 피트)                 |
| **OpenPorchSF**   | 오픈 현관 면적 (평방 피트)                 |
| **EnclosedPorch** | 밀폐된 현관 면적 (평방 피트)                |
| **3SsnPorch**     | 3계절용 현관 면적 (평방 피트)               |
| **ScreenPorch**   | 스크린 현관 면적 (평방 피트)                |
| **PoolArea**      | 수영장 면적 (평방 피트)                   |
| **PoolQC**        | 수영장 품질                           |
| **Fence**         | 울타리 품질                           |
| **MiscFeature**   | 기타 특수 기능                         |
| **MiscVal**       | 기타 기능의 금전적 가치                    |
| **MoSold**        | 판매 월                             |
| **YrSold**        | 판매 연도                            |
| **SaleType**      | 판매 유형                            |
| **SaleCondition** | 판매 조건                            |


In [6]:
train_df.head(3)

,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,968,20,RL,NaN,7390,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,7,2008,WD,Normal,135000
1,739,90,RL,60.0,10800,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,3,2009,WD,Alloca,179000
2,696,20,RL,54.0,13811,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,7,2006,WD,Normal,176000


In [7]:
print(train_df.isnull().sum())

Id                 0
MSSubClass         0
MSZoning           0
LotFrontage      218
LotArea            0
                ... 
MoSold             0
YrSold             0
SaleType           0
SaleCondition      0
SalePrice          0
Length: 81, dtype: int64


### 전처리

전처리에서 고민해야 할 것은 많습니다.

1. 결측치
2. 이상치

결측치는 처리하지 않으면 학습 자체가 불가할 수 있기 때문에, 최소한의 전처리로 결측치를 처리하겠습니다.

여러분들이라면 어떻게 결측치를 처리하면 좋을지 고민해보세요!

In [8]:
# 수치형 → 중앙값
train_df = train_df.fillna(train_df.median(numeric_only=True))
# 범주형 → 최빈값
for col in train_df.select_dtypes(include="object"):
    train_df[col] = train_df[col].fillna(train_df[col].mode()[0])

# 범주형 원 핫 인코딩
train_df = pd.get_dummies(train_df, drop_first=True)

C:\Users\user\AppData\Local\Temp\ipykernel_26092\3180690095.py:4: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in train_df.select_dtypes(include="object"):


In [9]:
# ===== 베이스라인 점수 =====
# 이유: 앞으로의 모든 개선을 비교할 기준선이 필요하다.
#      선생님 자료의 전처리를 그대로 쓴 상태에서 5겹 교차 검증 RMSE를 잰다.

X_base = train_df.drop("SalePrice", axis=1)
y_base = train_df["SalePrice"]
print("베이스라인 5겹 RMSE:", round(cv_rmse(LinearRegression(), X_base, y_base)))

베이스라인 5겹 RMSE: 38408


In [10]:
# ===== 원본 다시 불러오기 =====
# 이유: 위의 베이스라인 전처리 셀이 train_df를 이미 최빈값 채우기·원핫 인코딩으로 바꿔 버렸다.
#      개선 전처리는 원본에서 새로 시작해야 하므로 다시 읽는다.

train_df = pd.read_csv("./train.csv")
test_df = pd.read_csv("./test.csv")

In [11]:
# ===== 이상치 2채 제거 (train에서만) =====
# 이유 1: 면적 4,000평방피트 초과인데 30만 달러 미만인 집은 2채뿐이고,
#        둘 다 거래 조건이 Partial(완공 전 신축 주택 거래)이다.
#        다 지어지기 전의 거래라 기록된 가격이 집의 실제 시세가 아니다.
# 이유 2: 선형 회귀는 동떨어진 점에 크게 끌려가, "면적이 넓어도 집값이 안 오른다"는
#        쪽으로 기울기가 잘못 기운다. 큰 오차를 무겁게 보는 RMSE에서는 영향이 더 크다.
# 이유 3: 데이터 원작자(Dean De Cock)도 4,000평방피트 초과 집을 빼고 쓰기를 권한다.
# 참고: 넓고 비싼 나머지 2채는 면적-가격 관계를 따르므로 남긴다.
#      test에서는 빼지 않는다. test의 집은 모두 예측해서 제출해야 하기 때문이다.

outlier = (train_df["GrLivArea"] > 4000) & (train_df["SalePrice"] < 300000)
print(train_df.loc[outlier, ["Id", "GrLivArea", "SalePrice", "SaleCondition"]])  # 2채인지 확인

train_df = train_df[~outlier].reset_index(drop=True)

       Id  GrLivArea  SalePrice SaleCondition
64   1299       5642     160000       Partial
373   524       4676     184750       Partial


In [12]:
# ===== 빈 값을 채울 기준 준비 =====

# "시설 없음"을 뜻하는 열들
# 이유: 데이터 설명서에서 이 열들의 빈 값은 "해당 시설 없음"이다.
#      최빈값으로 채우면 PoolQC의 경우 수영장 없는 집 1,162채가
#      수영장 등급을 가진 집으로 바뀌어 잘못된 정보가 된다.
none_cols = ["PoolQC", "MiscFeature", "Alley", "Fence", "FireplaceQu",
             "GarageType", "GarageFinish", "GarageQual", "GarageCond",
             "BsmtQual", "BsmtCond", "BsmtExposure", "BsmtFinType1", "BsmtFinType2",
             "MasVnrType"]

# 나머지 빈 값(실제로 기록이 빠진 값)을 채울 값
# 이유: train에서만 계산하고 test에는 적용만 한다. test를 미리 보지 않고,
#      새 집이 한 채씩 들어오는 실제 상황과 조건을 맞추기 위해서다.
#      이상치를 뺀 뒤에 계산해야 이상치의 영향을 받지 않는다.
num_fill = train_df.median(numeric_only=True)
cat_fill = train_df.select_dtypes("object").mode().iloc[0]

C:\Users\user\AppData\Local\Temp\ipykernel_26092\532151388.py:17: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_fill = train_df.select_dtypes("object").mode().iloc[0]


In [13]:
# ===== 전처리 함수 =====
# 이유: train과 test에 똑같은 처리를 하기 위해 함수 하나로 만든다.
#      코드를 두 곳에 따로 쓰면 한쪽만 고치는 실수가 생긴다.

def prep(d):
    d = d.copy()

    # Id 빼기
    # 이유: 행 번호일 뿐 집값과 관계가 없다. 숫자라 그대로 두면 모델이
    #      우연히 맞아떨어진 패턴에 계수를 준다(잡음 피처와 같은 문제).
    d = d.drop(columns=["Id"])

    # MSSubClass를 글자로
    # 이유: 20, 60, 120은 주택 유형 코드 번호다. 숫자로 두면 "120은 20의 6배"로
    #      받아들이므로, 글자로 바꿔 유형마다 따로 계수를 배우게 한다.
    d["MSSubClass"] = d["MSSubClass"].astype(str)

    # "시설 없음" 열은 "None"으로
    d[none_cols] = d[none_cols].fillna("None")

    # 나머지 빈 값은 train의 중앙값(숫자)·최빈값(글자)으로
    d = d.fillna(num_fill).fillna(cat_fill)

        # ===== 대지 면적에 로그 =====
    # 이유 1: 대지 면적은 중앙값 9,446인데 최댓값이 164,660으로 극단적으로 치우쳐 있다.
    #        소수의 아주 넓은 땅에 선형 모델의 계수가 휘둘린다.
    # 이유 2: 땅은 넓을수록 1평방피트가 더해질 때의 가치가 줄어든다.
    #        로그를 씌우면 "얼마나 늘었나"가 아니라 "몇 배로 늘었나"로 보게 되어 이 성질과 맞는다.
    # 참고: GrLivArea에도 씌워 봤지만 교차 검증 RMSE가 오히려 나빠져서(21,746) 대지 면적에만 적용한다.
    # 위치 이유: 빈 값을 채운 뒤에 해야 하고, 함수 안에 넣어야 train과 test에 똑같이 적용된다.
    d["LotArea"] = np.log1p(d["LotArea"])

    # 글자 열을 0/1 열로 (모델은 숫자만 읽을 수 있다)
    d = pd.get_dummies(d).astype(float)
    return d


# ===== 적용 =====
test_id = test_df["Id"]          # Id를 피처에서 빼므로 제출용 Id는 미리 보관
y = train_df["SalePrice"]
X = prep(train_df.drop(columns=["SalePrice"]))

# test 열 구성을 train과 맞춘다
# 이유: 원핫 인코딩 후 test에 없는 범주의 열은 0으로 채우고, train에 없던 범주의 열은 버려야
#      모델이 학습 때와 같은 모양의 데이터를 받는다.
X_test = prep(test_df).reindex(columns=X.columns, fill_value=0)

print("개선 전처리 + 이상치 제거 5겹 RMSE:", round(cv_rmse(LinearRegression(), X, y)))

개선 전처리 + 이상치 제거 5겹 RMSE: 46280


In [14]:
from sklearn.compose import TransformedTargetRegressor

# ===== 집값에 로그를 씌워 학습 =====
# 이유 1: 집값은 오른쪽으로 길게 늘어진 분포라, 소수의 비싼 집이 학습을 끌고 간다.
# 이유 2: 비싼 집일수록 오차가 커진다(잔차가 나팔 모양). 로그를 씌우면 가격대별 오차 크기가 비슷해진다.
# 이유 3: 집값은 "몇 %" 단위로 움직이는데, 로그를 씌우면 비율 관계가 더하기 관계로 바뀌어
#        선형 회귀가 다루기 쉬워진다.

log_lr = TransformedTargetRegressor(
    regressor=LinearRegression(),
    func=np.log1p,          # 학습할 때: 가격 → log(1 + 가격)
    inverse_func=np.expm1   # 예측할 때: 다시 원래 가격으로 되돌림 (RMSE도 달러 단위로 계산됨)
)

print("+ 로그 가격 5겹 RMSE:", round(cv_rmse(log_lr, X, y)))

+ 로그 가격 5겹 RMSE: 23318


In [15]:
# ===== 과적합 확인 =====
# 이유: Lasso(정규화)는 과적합을 줄이는 도구다. 정말 과적합인지 먼저 확인해야
#      Lasso를 쓰는 근거가 생긴다. 학습용 오차가 교차 검증 오차보다 훨씬 작으면 과적합이다.

log_lr.fit(X, y)
train_rmse = np.sqrt(((y - log_lr.predict(X)) ** 2).mean())

print("열 개수:", X.shape[1], "/ 집 수:", X.shape[0])
print("학습용 RMSE:", round(train_rmse))
print("5겹 교차 검증 RMSE: 24358")

열 개수: 312 / 집 수: 1166
학습용 RMSE: 15537
5겹 교차 검증 RMSE: 24358


In [16]:
from sklearn.linear_model import LassoCV
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# ===== Lasso + 로그 가격 =====
# 이유 1: 열(약 290개)에 비해 집(1,166채)이 적어 과적합되기 쉽다.
#        Lasso는 계수가 커지는 데 벌점을 줘서 학습 데이터에 억지로 맞추는 것을 막는다.
# 이유 2: Lasso는 쓸모없거나 정보가 겹치는 열의 계수를 0으로 만들어 피처를 자동으로 고른다.

lasso = TransformedTargetRegressor(
    regressor=make_pipeline(
        # 스케일링 이유: Lasso의 벌점은 계수 크기로 매긴다. 면적(수천)과 0/1 열처럼
        #             단위가 다르면 벌점이 불공평하므로 먼저 단위를 맞춘다.
        StandardScaler(),

        # LassoCV 이유: 벌점 세기(alpha)를 직접 찍지 않고, 교차 검증으로 가장 좋은 값을 고른다.
        LassoCV(
            alphas=np.logspace(-4, -1, 30),   # 0.0001 ~ 0.1 사이 30개 후보
                                              # 로그 가격(약 10~13) 기준이라 alpha가 아주 작다
            max_iter=50000,                   # 열이 많아 계산이 오래 걸리므로 반복 횟수를 넉넉히
            random_state=42
        )
    ),
    func=np.log1p,
    inverse_func=np.expm1
)

print("Lasso + 로그 가격 5겹 RMSE:", round(cv_rmse(lasso, X, y)))

Lasso + 로그 가격 5겹 RMSE: 19979


In [17]:
# ===== Lasso가 고른 alpha와 남은 피처 수 =====
# 이유: 얼마나 센 벌점이 골라졌는지, 피처를 몇 개나 버렸는지 확인해 결과를 해석하기 위해서다.

lasso.fit(X, y)
lasso_model = lasso.regressor_[-1]      # 파이프라인의 마지막 단계(LassoCV)
print("고른 alpha:", lasso_model.alpha_)
print("남은 피처 수:", (lasso_model.coef_ != 0).sum(), "/", X.shape[1])

고른 alpha: 0.003562247890262444
남은 피처 수: 116 / 312


### 7단계. 모델이 설명하지 못하는 집을 학습에서 빼기
(1등을 되찾았던 단계입니다. 원본 노트북의 새 셀 A, B와 같은 내용입니다.)

In [18]:
# ===== 새 셀 A: 모델이 설명하지 못하는 집 찾기 =====
# 이유: 압류·급매(Abnorml), 완공 전 거래(Partial), 가족 간 거래(Family)처럼
#      가격이 집의 특징이 아니라 거래 사정으로 정해진 집이 있다.
#      면적·품질·동네로는 설명할 수 없는 가격이라, 여기에 맞추려 하면
#      평범한 집들에 대한 계산이 흐트러진다.

lasso.fit(X, y)

# 잔차 = 로그(실제 가격) - 로그(예측 가격)
# 로그로 재는 이유: 로그 차이는 "몇 % 틀렸나"에 가까워서 가격대와 상관없이 공평하게 비교된다.
resid = np.log1p(y) - np.log1p(lasso.predict(X))

# 기준: 잔차의 표준편차(평균적으로 틀리는 정도)의 3배
# 3배인 이유: 2.5배는 점수가 조금 더 낮지만 결과가 불안정했다(조각별 흔들림 2,177 → 2,879).
hard = resid.abs() >= 3 * resid.std()

print("빼는 집 수:", hard.sum(), "/", len(y))
print(train_df.loc[hard, "SaleCondition"].value_counts())   # 특수 거래가 많은지 확인

X_clean = X[~hard]
y_clean = y[~hard]

빼는 집 수: 19 / 1166
SaleCondition
Normal     9
Abnorml    5
Partial    4
Family     1
Name: count, dtype: int64


In [19]:
from sklearn.base import clone

# ===== 새 셀 B: 학습 묶음에서만 빼고 재는 교차 검증 =====
# 이유: 시험 보는 묶음에서도 어려운 집을 빼면 어려운 문제를 시험지에서 지운 셈이라
#      점수가 부풀려진다. 그래서 학습하는 4묶음에서만 빼고, 시험 묶음은 그대로 두고 잰다.

def cv_rmse_clean(model, X, y, k=3):
    errs = []
    for tr_idx, va_idx in kf.split(X):                 # 셀 6과 같은 5묶음
        X_tr, y_tr = X.iloc[tr_idx], y.iloc[tr_idx]
        m = clone(model).fit(X_tr, y_tr)               # 1) 한 번 학습해서
        r = np.log1p(y_tr) - np.log1p(m.predict(X_tr))
        keep = r.abs() < k * r.std()                   #    크게 틀리는 집 찾기
        m = clone(model).fit(X_tr[keep], y_tr[keep])   # 2) 그 집들을 빼고 다시 학습
        p = m.predict(X.iloc[va_idx])                  # 3) 시험 묶음은 그대로 채점
        errs.append(np.sqrt(((y.iloc[va_idx] - p) ** 2).mean()))
    return np.mean(errs)

print("어려운 집 제외 + Lasso 5겹 RMSE:", round(cv_rmse_clean(lasso, X, y)))   # 약 19,625

어려운 집 제외 + Lasso 5겹 RMSE: 19625


## 9단계. 그래디언트 부스팅 섞기 (실험용 복사본)

### 그래디언트 부스팅이란?
결정트리(해지 예측 실습에서 배운 "질문을 이어 가며 나누는 나무")를 **여러 그루 이어 붙인 모델**입니다.

1. 첫 번째 나무가 집값을 대충 예측합니다.
2. 두 번째 나무는 **첫 번째 나무가 틀린 만큼**을 맞히도록 배웁니다.
3. 세 번째 나무는 앞의 두 나무가 **아직도 틀린 만큼**을 배웁니다.
4. 이렇게 수천 그루가 조금씩 보충해서 최종 예측을 만듭니다.

시험 오답 노트와 같습니다. 틀린 문제만 골라 다시 공부하고, 또 틀린 문제를 다시 공부하는 식입니다.

### 왜 섞으면 좋아질 수 있나
- **Lasso(선형 모델)**: "피처 × 계수"를 더하는 방식이라 **직선 관계**를 잘 잡습니다.
- **그래디언트 부스팅(나무 모델)**: "품질이 높고, 면적도 넓으면" 같은 **조건 조합**을 스스로 찾습니다.

두 모델은 틀리는 집이 서로 다릅니다. 둘의 예측을 섞으면 한쪽의 실수를 다른 쪽이 메워 줄 수 있습니다.

### 미리 알아둘 점
train 안의 공정한 교차 검증에서 Lasso 80% + 부스팅 20%가 **약 200 정도** 좋았습니다(19,625 → 19,422).
조각마다의 흔들림(약 2,500)보다 작아서 **제출 점수가 좋아진다고 장담할 수 없습니다.** 8단계처럼 반대로 나올 수도 있습니다.

In [20]:
from sklearn.ensemble import GradientBoostingRegressor

# ===== 그래디언트 부스팅 모델 만들기 =====
gbm = TransformedTargetRegressor(
    regressor=GradientBoostingRegressor(
        n_estimators=3000,      # 나무 3,000그루를 이어 붙인다
        learning_rate=0.02,     # 나무 하나가 앞의 오차를 얼마나 고칠지. 작게 해서 조금씩 고친다
                                # 이유: 한 번에 크게 고치면 문제집에 과하게 맞춰진다(과적합)
        max_depth=4,            # 나무 하나가 던질 수 있는 질문은 4번까지
                                # 이유: 깊으면 외우기 쉽고, 얕으면 조합을 못 찾는다
        subsample=0.7,          # 나무마다 집의 70%만 무작위로 보고 배운다
        max_features="sqrt",    # 질문을 고를 때 열 일부만 후보로 본다
                                # 이유(두 줄 모두): 나무들이 서로 다른 것을 보게 해서 외우기를 막는다
        min_samples_leaf=10,    # 잎 하나에 집이 최소 10채는 있어야 한다
                                # 이유: 집 한두 채에만 맞춘 규칙을 만들지 못하게 한다
        loss="huber",           # 크게 틀린 집에는 덜 끌려가는 오차 계산 방식
                                # 이유: 거래 사정으로 가격이 튄 집에 휘둘리지 않게 한다
        random_state=42         # 무작위 선택을 고정해서 매번 같은 결과가 나오게 한다
    ),
    func=np.log1p,              # Lasso와 똑같이 집값에 로그를 씌워 배우고
    inverse_func=np.expm1       # 예측할 때 달러로 되돌린다
)

In [21]:
# ===== 섞는 비율 정하기 (공정한 교차 검증) =====
# 이유: 몇 대 몇으로 섞을지 test를 보고 정하면 안 된다. train 안에서만 정한다.
#      7단계와 똑같이, 학습 묶음에서만 어려운 집을 빼고 시험 묶음은 그대로 채점한다.
#      나무 3,000그루를 5번 학습하므로 몇 분 걸린다.

pred_lasso, pred_gbm, answers = [], [], []

for tr_idx, va_idx in kf.split(X):
    X_tr, y_tr = X.iloc[tr_idx], y.iloc[tr_idx]

    # 학습 묶음에서 어려운 집 찾아 빼기 (7단계와 같은 방법)
    m = clone(lasso).fit(X_tr, y_tr)
    r = np.log1p(y_tr) - np.log1p(m.predict(X_tr))
    keep = r.abs() < 3 * r.std()

    # 두 모델을 같은 데이터로 학습하고, 같은 시험 묶음을 예측
    pred_lasso.append(clone(lasso).fit(X_tr[keep], y_tr[keep]).predict(X.iloc[va_idx]))
    pred_gbm.append(clone(gbm).fit(X_tr[keep], y_tr[keep]).predict(X.iloc[va_idx]))
    answers.append(y.iloc[va_idx].values)

# 섞는 비율별 점수 비교
# w = Lasso 비율. 1.0이면 Lasso만, 0.0이면 부스팅만
for w in [1.0, 0.9, 0.8, 0.7, 0.0]:
    errs = [np.sqrt(((a - (w * pl + (1 - w) * pg)) ** 2).mean())
            for pl, pg, a in zip(pred_lasso, pred_gbm, answers)]
    print(f"Lasso {w:.0%} + 부스팅 {1 - w:.0%}: 5겹 RMSE {np.mean(errs):,.0f}")

Lasso 100% + 부스팅 0%: 5겹 RMSE 19,625
Lasso 90% + 부스팅 10%: 5겹 RMSE 19,473
Lasso 80% + 부스팅 20%: 5겹 RMSE 19,422
Lasso 70% + 부스팅 30%: 5겹 RMSE 19,471
Lasso 0% + 부스팅 100%: 5겹 RMSE 22,224


### 모델 설정 및 학습

In [ ]:
# 5겹 교차 검증으로 이미 대신하였기 때문에 이 셀은 건너뜀
#from sklearn.model_selection import train_test_split

#X = train_df.drop("SalePrice", axis=1)
#y = train_df["SalePrice"]

#X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
#from sklearn.linear_model import LinearRegression

#lr = LinearRegression()
#lr.fit(X_train, y_train)

In [ ]:
#from sklearn.metrics import mean_squared_error
#import numpy as np

#y_pred = lr.predict(X_val)
#rmse = np.sqrt(mean_squared_error(y_val, y_pred))
#print("Validation RMSE:", rmse)

### 테스트 데이터 예측 및 제출

In [ ]:
# 동일한 전처리(기존 선생님께서 주신 방법으로 이건 사용하지 않겠습니다.)
#test_df = test_df.fillna(test_df.median(numeric_only=True))
#for col in test_df.select_dtypes(include="object"):
#    test_df[col] = test_df[col].fillna(test_df[col].mode()[0])

#test_df = pd.get_dummies(test_df, drop_first=True)

# train/test 컬럼 차이 맞추기
#test_df = test_df.reindex(columns=X.columns, fill_value=0)

# 예측
#test_pred = lr.predict(test_df)

In [22]:
# ===== 최종 학습과 test 예측 (섞기 버전) =====
# 이유: 비율은 위 셀에서 train 안의 교차 검증으로 정했다(Lasso 80% + 부스팅 20%).
#      두 모델 모두 어려운 집을 뺀 train 전체(X_clean, y_clean)로 다시 학습한다.

W = 0.8   # Lasso 비율. 위 셀 결과를 보고 가장 낮은 비율로 바꿔도 된다

final_lasso = clone(lasso).fit(X_clean, y_clean)
final_gbm = clone(gbm).fit(X_clean, y_clean)

test_pred = W * final_lasso.predict(X_test) + (1 - W) * final_gbm.predict(X_test)

In [23]:
# Id는 피처에서 뺐으므로 셀 17에서 보관해 둔 test_id를 쓴다
submission = pd.DataFrame({
    "Id": test_id,
    "SalePrice": test_pred
})
submission.to_csv("submission_step9_blend.csv", index=False)   # 원본 노트북의 제출 파일을 덮어쓰지 않도록 새 이름
print(submission.shape)   # (292, 2)가 나와야 한다

(292, 2)


In [ ]:
#기존에 있던 셀로 안 쓰겠습니다.
# submission = pd.DataFrame({
#    "Id": test_df["Id"],
#    "SalePrice": test_pred
#})
#submission.to_csv("submission.csv", index=False)